In [ ]:
# =========================
# Cell 0. 环境配置
# =========================

import sys
import subprocess
import importlib.util


def pip_install(package: str):
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        package,
    ])


# OpenCV 在 kaggle 环境里不一定有，使用 headless 版本即可
if importlib.util.find_spec("cv2") is None:
    pip_install("opencv-python-headless")

# 常规依赖
for pkg, import_name in [
    ("pillow", "PIL"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("tqdm", "tqdm"),
    ("kaggle-benchmarks", "kaggle_benchmarks"),
    ("jupyter_bokeh", "jupyter_bokeh"),
]:
    if importlib.util.find_spec(import_name) is None:
        pip_install(pkg)


# kaggle-benchmarks 0.5.0 生成的 protobuf 代码要求 5.29.6。
# 必须在 import google.protobuf / kaggle_benchmarks 之前安装，否则当前进程会继续使用旧 runtime。
pip_install("protobuf==5.29.6")

print("环境依赖检查完成")

# 运行模式说明：
# 1. Kaggle Benchmark / 已认证环境：自动跑片段理解并导出 candidate_clips.json。
# 2. 普通 Kaggle kernel / VS Code：如果没有 kbench.llm，会保留最后的 %choose 入口给 Benchmark UI 跑。
# 3. 重新跑导出 cell 时，只要结果 JSONL 已存在，也会补导出脚本。
RUN_VIDEO_OVERVIEW_BENCHMARK = False
RUN_SEGMENT_BENCHMARK = True

# 片段窗口配置：短窗口 + 重叠，比 30s batch 更接近剪辑节奏。
SEGMENT_SECONDS = 14.0
SEGMENT_STRIDE_SECONDS = 10.0
MAX_SEGMENTS_PER_VIDEO = 48
SAMPLE_FRAMES_PER_SEGMENT = 8
MIN_EVENT_SCORE = 7.0


In [ ]:
# =========================
# Cell 1. 导入库与扫描视频
# =========================

import os
import cv2
import json
import math
import hashlib
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List

import pandas as pd
import numpy as np
from tqdm import tqdm

# pyright: reportMissingImports=false
# kaggle_benchmarks 是 Kaggle Benchmark 运行时依赖；VS Code 本地 kernel 可能会误报 missing import。
import kaggle_benchmarks as kbench  # type: ignore[reportMissingImports]
from kaggle_benchmarks.content_types import images  # type: ignore[reportMissingImports]

BENCHMARK_LLM = getattr(kbench, "llm", None)
if BENCHMARK_LLM is None:
    print("未检测到 Kaggle Benchmark LLM 代理；当前只生成 Benchmark task，不自动跑模型。")
    print("在 Kaggle Benchmark UI 中使用最后一格 %choose narrato_segment_understanding 选择模型运行。")
else:
    print("已检测到 Kaggle Benchmark LLM 代理。")


INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
STORYBOARD_DIR = WORK_ROOT / "storyboards"
RESULT_DIR = WORK_ROOT / "results"

STORYBOARD_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi", ".webm"}


def find_video_files(input_root: Path = INPUT_ROOT) -> List[str]:
    video_files = []
    for p in input_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
            video_files.append(str(p))
    return sorted(video_files)


video_files = find_video_files()

print(f"找到视频数量: {len(video_files)}")
for p in video_files[:30]:
    print(p)

In [ ]:
# =========================
# Cell 2. 视频元信息读取
# =========================

def probe_video(video_path: str) -> dict:
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError(f"无法打开视频: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    duration = total_frames / fps if fps and fps > 0 else 0.0

    cap.release()

    return {
        "video_path": video_path,
        "fps": fps,
        "total_frames": total_frames,
        "duration": duration,
        "width": width,
        "height": height,
    }


if video_files:
    info = probe_video(video_files[0])
    print(json.dumps(info, indent=2, ensure_ascii=False))

In [ ]:
# =========================
# Cell 3. 抽关键帧并拼成 storyboard
# =========================

def safe_video_id(video_path: str) -> str:
    name = Path(video_path).stem
    digest = hashlib.md5(video_path.encode("utf-8")).hexdigest()[:8]
    return f"{name}_{digest}"


def read_frame_at_time(cap, fps: float, timestamp: float):
    frame_idx = int(timestamp * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

    ok, frame = cap.read()
    if not ok:
        return None

    return frame


def resize_keep_ratio(frame, target_width: int = 320):
    h, w = frame.shape[:2]
    if w <= 0 or h <= 0:
        return frame

    scale = target_width / w
    target_height = max(1, int(h * scale))
    return cv2.resize(frame, (target_width, target_height))


def draw_timestamp(frame, timestamp: float):
    label = f"{timestamp:.1f}s"

    cv2.rectangle(frame, (0, 0), (100, 32), (0, 0, 0), -1)
    cv2.putText(
        frame,
        label,
        (8, 23),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )

    return frame


def make_video_storyboard(
    video_path: str,
    output_dir: Path = STORYBOARD_DIR,
    num_frames: int = 8,
    cell_width: int = 320,
    cols: int = 4,
) -> dict:
    """
    从视频中均匀抽 num_frames 帧，拼成一张 storyboard。
    返回 storyboard 路径和抽帧时间戳。
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"无法打开视频: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps <= 0 or total_frames <= 0:
        cap.release()
        raise RuntimeError(
            f"视频元信息异常: {video_path}, fps={fps}, total_frames={total_frames}"
        )

    duration = total_frames / fps

    # 避开首尾，减少黑帧、片头片尾干扰
    timestamps = [
        duration * (i + 1) / (num_frames + 1)
        for i in range(num_frames)
    ]

    frames = []
    valid_timestamps = []

    for t in timestamps:
        frame = read_frame_at_time(cap, fps, t)
        if frame is None:
            continue

        frame = resize_keep_ratio(frame, target_width=cell_width)
        frame = draw_timestamp(frame, t)

        frames.append(frame)
        valid_timestamps.append(t)

    cap.release()

    if not frames:
        raise RuntimeError(f"没有抽出任何有效帧: {video_path}")

    # 统一高度
    max_h = max(f.shape[0] for f in frames)
    normalized = []

    for f in frames:
        h, w = f.shape[:2]
        if h < max_h:
            pad_h = max_h - h
            f = cv2.copyMakeBorder(
                f,
                0,
                pad_h,
                0,
                0,
                cv2.BORDER_CONSTANT,
                value=(255, 255, 255),
            )
        normalized.append(f)

    rows = math.ceil(len(normalized) / cols)

    blank = np.ones_like(normalized[0]) * 255
    grid_rows = []

    for r in range(rows):
        row_imgs = []
        for c in range(cols):
            idx = r * cols + c
            if idx < len(normalized):
                row_imgs.append(normalized[idx])
            else:
                row_imgs.append(blank.copy())

        grid_rows.append(cv2.hconcat(row_imgs))

    storyboard = cv2.vconcat(grid_rows)

    video_id = safe_video_id(video_path)
    out_path = output_dir / f"{video_id}_storyboard.jpg"

    ok = cv2.imwrite(str(out_path), storyboard)
    if not ok:
        raise RuntimeError(f"storyboard 写入失败: {out_path}")

    return {
        "video_path": video_path,
        "storyboard_path": str(out_path),
        "timestamps": valid_timestamps,
        "duration": duration,
        "fps": fps,
        "total_frames": total_frames,
    }


# 测试生成第一条视频的 storyboard
if video_files:
    test_storyboard = make_video_storyboard(video_files[0], num_frames=8)
    print(json.dumps(test_storyboard, indent=2, ensure_ascii=False))

In [ ]:
# =========================
# Cell 4. 定义 VLM 输出结构
# =========================

@dataclass
class NarratoVideoUnderstanding:
    summary: str
    main_objects: List[str]
    main_actions: List[str]
    scene: str
    screen_text: str
    ost: int
    ost_reason: str
    narration: str


@dataclass
class NarratoSegmentUnderstanding:
    summary: str
    event_type: str
    score: float
    confidence: float
    visual_evidence: str
    highlight_reason: str
    main_objects: List[str]
    main_actions: List[str]
    scene: str
    screen_text: str
    ost: int
    ost_reason: str
    narration: str
    recommended_start_sec: float
    recommended_end_sec: float


In [ ]:
# =========================
# Cell 5. 定义 Benchmark Task：修正版
# =========================

REPORT_JSONL = RESULT_DIR / "narrato_video_understanding_report.jsonl"


def append_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


@kbench.task(
    name="NarratoAI Video Frame Understanding",
    description=(
        "Evaluate whether a multimodal LLM can understand sampled video frames "
        "and generate NarratoAI-style script metadata."
    ),
)
def narrato_video_understanding(llm, video_path: str) -> bool:
    """
    Kaggle Benchmark Task:
    输入视频路径 → 抽关键帧 → 拼 storyboard → VLM 理解 → 输出 NarratoAI script 元信息

    注意：
    - task 返回 bool，方便 Kaggle Benchmark 记录通过/失败。
    - 详细结果写入 /kaggle/working/results/narrato_video_understanding_report.jsonl
    """

    storyboard_info = make_video_storyboard(
        video_path=video_path,
        output_dir=STORYBOARD_DIR,
        num_frames=8,
        cell_width=320,
        cols=4,
    )

    img = images.from_path(storyboard_info["storyboard_path"])

    prompt = """
你正在为 NarratoAI 做视频理解。

下面这张图片是同一个视频按时间顺序抽取的关键帧拼图。
每个小图左上角是该帧在原视频中的时间戳。

请根据画面内容输出结构化结果。

你需要完成：

1. summary:
   用中文概括这个视频片段发生了什么。

2. main_objects:
   列出主要人物、物体、地点元素。

3. main_actions:
   列出主要动作变化。

4. scene:
   判断场景类型，例如室内、街道、会议、厨房、游戏画面、影视片段等。

5. screen_text:
   如果画面中有可见文字，请提取主要文字。
   如果没有可见文字，返回空字符串。

6. ost:
   判断 NarratoAI 的 OST 策略：
   - 0：纯 AI 解说，原声静音
   - 1：保留原声，不加旁白
   - 2：保留低音量原声，同时叠加 AI 解说

   判断规则：
   - 普通画面展示、动作过程、风景、无明显口播：优先 ost=0。
   - 明显采访、演讲、人物对话、口型强烈依赖原声：ost=1。
   - 有环境氛围、音乐场景、现场感，但仍适合加旁白：ost=2。
   - 仅凭画面无法确认有人说话时，不要强行设为 ost=1。

7. ost_reason:
   用一句话解释为什么选择这个 OST。

8. narration:
   生成一句适合中文视频解说的旁白。
   要自然、简洁、有叙事感，不要超过 40 个中文字符。

要求：
- 不要编造画面中不存在的信息。
- 不要输出 Markdown。
- 不要输出多余解释。
"""

    result = llm.prompt(
        prompt,
        image=img,
        schema=NarratoVideoUnderstanding,
    )

    # -------------------------
    # 基础硬规则校验
    # -------------------------
    kbench.assertions.assert_true(
        result.ost in [0, 1, 2],
        expectation="ost 必须是 0、1、2 之一。",
    )

    kbench.assertions.assert_true(
        len(result.summary.strip()) >= 5,
        expectation="summary 不能过短。",
    )

    kbench.assertions.assert_true(
        len(result.narration.strip()) >= 5,
        expectation="narration 不能过短。",
    )

    kbench.assertions.assert_true(
        len(result.narration.strip()) <= 80,
        expectation="narration 不应过长，避免 TTS 超出片段时长。",
    )

    # -------------------------
    # 保存详细结果
    # -------------------------
    record = {
        "video_path": video_path,
        "storyboard_path": storyboard_info["storyboard_path"],
        "duration": storyboard_info["duration"],
        "timestamps": storyboard_info["timestamps"],
        "summary": result.summary,
        "main_objects": result.main_objects,
        "main_actions": result.main_actions,
        "scene": result.scene,
        "screen_text": result.screen_text,
        "ost": result.ost,
        "ost_reason": result.ost_reason,
        "narration": result.narration,
    }

    append_jsonl(REPORT_JSONL, record)

    return True

In [ ]:
# =========================
# Cell 6. 单视频测试：仅 DEBUG 模式运行
# =========================

if RUN_VIDEO_OVERVIEW_BENCHMARK and BENCHMARK_LLM is not None:
    from IPython.display import Image, display
    import json

    if not video_files:
        raise RuntimeError("没有找到视频文件，请检查 /kaggle/input 是否已经添加视频数据集。")

    # 清空旧 report，避免混在一起
    if REPORT_JSONL.exists():
        REPORT_JSONL.unlink()

    video_path = video_files[0]
    print("测试视频:", video_path)

    # 注意：run 返回的是 Run 对象，不是 dict
    single_run = narrato_video_understanding.run(
        BENCHMARK_LLM,
        video_path=video_path,
    )

    print("Run 对象类型:", type(single_run))
    print("任务已执行。详细结果应写入:", REPORT_JSONL)

    # 检查 task 是否真的写出了 JSONL
    if not REPORT_JSONL.exists():
        raise RuntimeError(
            "没有生成 REPORT_JSONL。说明你还没有替换 Cell 5，"
            "或者 narrato_video_understanding() 里没有 append_jsonl(REPORT_JSONL, record)。"
        )

    # 读取最后一条详细结果
    with open(REPORT_JSONL, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    if not lines:
        raise RuntimeError("REPORT_JSONL 是空的，说明 task 没有成功写入结果。")

    last_record = json.loads(lines[-1])

    print("\n===== 视频理解结果 =====")
    print(json.dumps(last_record, indent=2, ensure_ascii=False))

    print("\n===== Storyboard 预览 =====")
    display(Image(filename=last_record["storyboard_path"]))
elif RUN_VIDEO_OVERVIEW_BENCHMARK:
    print("RUN_VIDEO_OVERVIEW_BENCHMARK=True，但当前环境没有 kbench.llm，跳过 Cell 6 单视频测试。")
else:
    print("RUN_VIDEO_OVERVIEW_BENCHMARK=False，跳过 Cell 6 单视频测试。")


In [ ]:
# =========================
# Cell 7. 批量评测 10 个视频
# =========================

if RUN_VIDEO_OVERVIEW_BENCHMARK and BENCHMARK_LLM is not None:
    import pandas as pd
    import json
    from IPython.display import display

    # 清空旧 report，避免和 C6 的单视频结果混在一起
    if REPORT_JSONL.exists():
        REPORT_JSONL.unlink()

    # 先不要全量跑，先跑前 10 个
    sample_video_files = video_files[:10]

    eval_df = pd.DataFrame({
        "video_path": sample_video_files,
    })

    print("本次评测视频数量:", len(eval_df))
    display(eval_df)

    runs = narrato_video_understanding.evaluate(
        llm=[BENCHMARK_LLM],
        evaluation_data=eval_df,
        max_attempts=2,
        retry_delay=5,
        remove_run_files=False,
    )

    # Kaggle Benchmark 自己的运行结果
    runs_df = runs.as_dataframe()
    print("===== Kaggle Run 结果 =====")
    display(runs_df)

    # 读取我们自己写出的详细理解结果
    records = []

    if REPORT_JSONL.exists():
        with open(REPORT_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))

    details_df = pd.DataFrame(records)

    print("===== NarratoAI 视频理解详细结果 =====")
    display(details_df)

    # 保存结果
    csv_path = RESULT_DIR / "narrato_video_understanding_details.csv"
    json_path = RESULT_DIR / "narrato_video_understanding_details.json"

    details_df.to_csv(csv_path, index=False)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print("详细 CSV:", csv_path)
    print("详细 JSON:", json_path)
elif RUN_VIDEO_OVERVIEW_BENCHMARK:
    print("RUN_VIDEO_OVERVIEW_BENCHMARK=True，但当前环境没有 kbench.llm，跳过 Cell 7 批量评测视频。")
else:
    print("RUN_VIDEO_OVERVIEW_BENCHMARK=False，跳过 Cell 7 批量评测视频。")


In [ ]:
# =========================
# 检查 C7 是否写出了结果
# =========================

if RUN_VIDEO_OVERVIEW_BENCHMARK:
    print("REPORT_JSONL:", REPORT_JSONL)
    print("exists:", REPORT_JSONL.exists())

    if REPORT_JSONL.exists():
        with open(REPORT_JSONL, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]
        
        print("写入记录数:", len(lines))
        
        for i, line in enumerate(lines):
            print(f"\n===== record {i} =====")
            record = json.loads(line)
            print(json.dumps(record, indent=2, ensure_ascii=False))
    else:
        print("还没有生成 report 文件")
else:
    print("RUN_VIDEO_OVERVIEW_BENCHMARK=False，跳过 C7 结果检查。")


In [ ]:
# =========================
# Cell 8. 生成视频片段索引：短窗口 + 重叠
# =========================

import math
import pandas as pd
from pathlib import Path


def build_video_segments(
    video_files,
    segment_seconds: float = SEGMENT_SECONDS,
    stride_seconds: float = SEGMENT_STRIDE_SECONDS,
    max_segments_per_video: int | None = MAX_SEGMENTS_PER_VIDEO,
):
    rows = []

    for video_path in video_files:
        info = probe_video(video_path)
        duration = float(info["duration"])

        if duration <= 0:
            continue

        segment_index = 0
        start_sec = 0.0
        while start_sec < duration:
            end_sec = min(start_sec + segment_seconds, duration)
            if end_sec - start_sec >= 3:
                rows.append({
                    "video_path": video_path,
                    "segment_index": segment_index,
                    "start_sec": round(start_sec, 2),
                    "end_sec": round(end_sec, 2),
                    "duration": round(end_sec - start_sec, 2),
                })
                segment_index += 1

            if max_segments_per_video is not None and segment_index >= max_segments_per_video:
                break
            start_sec += stride_seconds

    return pd.DataFrame(rows)


segment_df = build_video_segments(video_files)

print("视频数量:", len(video_files))
print("片段数量:", len(segment_df))
display(segment_df.head(30))


In [ ]:
# =========================
# Cell 9. 针对单个片段抽帧并生成 storyboard
# =========================

def make_segment_storyboard(
    video_path: str,
    start_sec: float,
    end_sec: float,
    segment_index: int,
    output_dir: Path = STORYBOARD_DIR,
    num_frames: int = SAMPLE_FRAMES_PER_SEGMENT,
    cell_width: int = 360,
    cols: int = 4,
) -> dict:
    """
    对视频的 [start_sec, end_sec] 区间抽帧，生成 storyboard。
    不把时间戳画进图里，避免 VLM 把采样标签识别成 screen_text。
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"无法打开视频: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps <= 0 or total_frames <= 0:
        cap.release()
        raise RuntimeError(f"视频元信息异常: {video_path}, fps={fps}, total_frames={total_frames}")

    video_duration = total_frames / fps
    start_sec = max(0.0, float(start_sec))
    end_sec = min(float(end_sec), video_duration)

    if end_sec <= start_sec:
        cap.release()
        raise RuntimeError(f"非法片段: start={start_sec}, end={end_sec}")

    seg_duration = end_sec - start_sec

    timestamps = [
        start_sec + seg_duration * (i + 1) / (num_frames + 1)
        for i in range(num_frames)
    ]

    frames = []
    valid_timestamps = []

    for t in timestamps:
        frame = read_frame_at_time(cap, fps, t)
        if frame is None:
            continue
        frame = resize_keep_ratio(frame, target_width=cell_width)
        frames.append(frame)
        valid_timestamps.append(round(t, 2))

    cap.release()

    if not frames:
        raise RuntimeError(f"没有抽出有效帧: {video_path}, start={start_sec}, end={end_sec}")

    max_h = max(f.shape[0] for f in frames)
    normalized = []
    for f in frames:
        h, w = f.shape[:2]
        if h < max_h:
            f = cv2.copyMakeBorder(f, 0, max_h - h, 0, 0, cv2.BORDER_CONSTANT, value=(255, 255, 255))
        normalized.append(f)

    rows = math.ceil(len(normalized) / cols)
    blank = np.ones_like(normalized[0]) * 255
    grid_rows = []
    for r in range(rows):
        row_imgs = []
        for c in range(cols):
            idx = r * cols + c
            row_imgs.append(normalized[idx] if idx < len(normalized) else blank.copy())
        grid_rows.append(cv2.hconcat(row_imgs))

    storyboard = cv2.vconcat(grid_rows)

    video_id = safe_video_id(video_path)
    out_path = output_dir / f"{video_id}_seg{segment_index:04d}_{start_sec:.1f}_{end_sec:.1f}.jpg"

    ok = cv2.imwrite(str(out_path), storyboard)
    if not ok:
        raise RuntimeError(f"storyboard 写入失败: {out_path}")

    return {
        "video_path": video_path,
        "storyboard_path": str(out_path),
        "segment_index": segment_index,
        "start_sec": round(start_sec, 2),
        "end_sec": round(end_sec, 2),
        "duration": round(seg_duration, 2),
        "timestamps": valid_timestamps,
    }


In [ ]:
# =========================
# Cell 10. 片段级视频理解 Task：事件评分版
# =========================

SEGMENT_REPORT_JSONL = RESULT_DIR / "narrato_segment_understanding_report.jsonl"

EVENT_TYPES = {
    "death_fail",
    "strong_reaction",
    "coop_command",
    "puzzle_progress",
    "live_interaction",
    "transition",
    "low_value",
}


def clamp(value, low, high):
    return max(low, min(high, value))


@kbench.task(
    name="NarratoAI Segment Event Understanding",
    description=(
        "Evaluate whether a multimodal LLM can identify edit-worthy events "
        "from sampled video segment frames."
    ),
)
def narrato_segment_understanding(
    llm,
    video_path: str,
    segment_index: int,
    start_sec: float,
    end_sec: float,
) -> bool:
    storyboard_info = make_segment_storyboard(
        video_path=video_path,
        start_sec=start_sec,
        end_sec=end_sec,
        segment_index=int(segment_index),
        output_dir=STORYBOARD_DIR,
    )

    img = images.from_path(storyboard_info["storyboard_path"])

    prompt = f"""
你正在为 NarratoAI 做游戏/直播短视频剪辑识别。

当前视频片段时间范围：
start_sec = {start_sec}
end_sec = {end_sec}

下面这张图片是该片段内部按时间顺序抽取的关键帧拼图。
图片中如果出现采样标签、时间戳或帧编号，不属于原始视频画面文字，必须忽略。

你的任务不是写普通摘要，而是判断这段是否值得剪进短视频。

请输出结构化结果：

1. summary: 用中文概括片段发生了什么。
2. event_type: 只能是 death_fail / strong_reaction / coop_command / puzzle_progress / live_interaction / transition / low_value。
3. score: 0-10 的剪辑价值分。低价值赶路/等待/菜单操作给 0-3；有明确视觉事件给 7+。
4. confidence: 0-1 的置信度。
5. visual_evidence: 必须写清楚可见画面证据。没有画面证据时写空字符串，并把 event_type 设为 low_value。
6. highlight_reason: 为什么它值得或不值得剪。
7. main_objects: 主要人物、物体、地点元素。
8. main_actions: 主要动作变化。
9. scene: 场景类型，例如游戏画面、菜单、过场、室内、影视片段等。
10. screen_text: 只提取原始视频画面真实文字，不要提取 storyboard 标签。
11. ost: NarratoAI 音频策略：0=纯 AI 解说，1=保留原声不加旁白，2=保留低音量原声同时叠加 AI 解说。
12. ost_reason: 一句话解释 OST。
13. narration: 如果适合加旁白，写一句 18-38 个中文字符的自然解说；如果必须保留原声，可写空字符串。
14. recommended_start_sec / recommended_end_sec: 推荐剪辑入点和出点，必须在当前片段范围内；好片段保留前后 1-3 秒。

重要规则：
- 不要只因为字幕或文字看起来有趣就判高分，必须有视觉证据。
- 菜单、加载、普通行走、无变化画面通常是 low_value。
- 死亡、失败、掉落、强反应、关键机关推进、明显合作动作才适合高分。
- 不要输出 Markdown，不要输出多余解释。
"""

    result = llm.prompt(
        prompt,
        image=img,
        schema=NarratoSegmentUnderstanding,
    )

    event_type = str(result.event_type).strip()
    if event_type not in EVENT_TYPES:
        event_type = "low_value"

    score = clamp(float(result.score), 0.0, 10.0)
    confidence = clamp(float(result.confidence), 0.0, 1.0)
    rec_start = clamp(float(result.recommended_start_sec), float(start_sec), float(end_sec))
    rec_end = clamp(float(result.recommended_end_sec), float(start_sec), float(end_sec))
    if rec_end <= rec_start:
        rec_start, rec_end = float(start_sec), float(end_sec)

    kbench.assertions.assert_true(event_type in EVENT_TYPES, expectation="event_type 必须合法。")
    kbench.assertions.assert_true(result.ost in [0, 1, 2], expectation="ost 必须是 0、1、2 之一。")
    kbench.assertions.assert_true(len(result.summary.strip()) >= 5, expectation="summary 不能过短。")
    kbench.assertions.assert_true(0.0 <= score <= 10.0, expectation="score 必须在 0-10。")
    kbench.assertions.assert_true(0.0 <= confidence <= 1.0, expectation="confidence 必须在 0-1。")

    record = {
        "video_path": video_path,
        "segment_index": int(segment_index),
        "start": float(start_sec),
        "end": float(end_sec),
        "duration": float(end_sec) - float(start_sec),
        "recommended_start": round(rec_start, 2),
        "recommended_end": round(rec_end, 2),
        "recommended_duration": round(rec_end - rec_start, 2),
        "event_type": event_type,
        "score": score,
        "confidence": confidence,
        "visual_evidence": result.visual_evidence,
        "highlight_reason": result.highlight_reason,
        "ost": int(result.ost),
        "text": result.narration,
        "summary": result.summary,
        "scene": result.scene,
        "main_objects": result.main_objects,
        "main_actions": result.main_actions,
        "screen_text": result.screen_text,
        "ost_reason": result.ost_reason,
        "storyboard": storyboard_info["storyboard_path"],
        "sample_timestamps": storyboard_info["timestamps"],
        "source_video": video_path,
    }

    append_jsonl(SEGMENT_REPORT_JSONL, record)
    return True


In [ ]:
# =========================
# Cell 11. 批量跑片段级理解
# =========================

SEGMENT_BENCHMARK_RAN = False

if RUN_SEGMENT_BENCHMARK and BENCHMARK_LLM is not None:
    from IPython.display import display
    import json
    import pandas as pd

    if SEGMENT_REPORT_JSONL.exists():
        SEGMENT_REPORT_JSONL.unlink()

    eval_segment_df = segment_df[[
        "video_path",
        "segment_index",
        "start_sec",
        "end_sec",
    ]].copy()

    print("本次片段数量:", len(eval_segment_df))
    display(eval_segment_df.head(30))

    runs = narrato_segment_understanding.evaluate(
        llm=[BENCHMARK_LLM],
        evaluation_data=eval_segment_df,
        max_attempts=2,
        retry_delay=5,
        remove_run_files=False,
    )

    runs_df = runs.as_dataframe()
    print("===== Kaggle Run 结果 =====")
    display(runs_df)

    records = []
    if SEGMENT_REPORT_JSONL.exists():
        with open(SEGMENT_REPORT_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))

    segment_details_df = pd.DataFrame(records)

    print("===== 片段级事件理解结果 =====")
    if segment_details_df.empty:
        print("没有读到片段识别记录，请检查 Benchmark run 是否成功写入 JSONL。")
    else:
        display(segment_details_df.sort_values(["source_video", "recommended_start"]).head(80))

    segment_csv_path = RESULT_DIR / "narrato_segment_understanding_details.csv"
    segment_json_path = RESULT_DIR / "narrato_segment_understanding_details.json"

    segment_details_df.to_csv(segment_csv_path, index=False)

    with open(segment_json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print("片段级 CSV:", segment_csv_path)
    print("片段级 JSON:", segment_json_path)
    SEGMENT_BENCHMARK_RAN = True
elif RUN_SEGMENT_BENCHMARK:
    print("RUN_SEGMENT_BENCHMARK=True，但当前环境没有 kbench.llm，跳过自动片段理解。")
    print("这是正常状态：普通 kernel 负责产出 .task.json，Benchmark UI 负责选择模型并消耗额度。")
else:
    print("RUN_SEGMENT_BENCHMARK=False，跳过片段级批量理解。")


In [ ]:
# =========================
# Cell 12. 生成 candidate_clips.json + NarratoAI script JSON
# =========================

from pathlib import Path


def ts(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    ms = int(round((seconds - math.floor(seconds)) * 1000))
    total = int(math.floor(seconds))
    if ms >= 1000:
        total += 1
        ms -= 1000
    h = total // 3600
    m = (total % 3600) // 60
    s = total % 60
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


if SEGMENT_REPORT_JSONL.exists():
    records = []

    with open(SEGMENT_REPORT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    if not records:
        raise RuntimeError(f"{SEGMENT_REPORT_JSONL} 存在但没有有效 JSONL 记录。")

    dedup = {}
    for r in records:
        key = (
            r["source_video"],
            round(float(r.get("recommended_start", r["start"])), 2),
            round(float(r.get("recommended_end", r["end"])), 2),
        )
        old = dedup.get(key)
        if old is None or float(r.get("score", 0)) > float(old.get("score", 0)):
            dedup[key] = r

    records = sorted(
        dedup.values(),
        key=lambda x: (x["source_video"], -float(x.get("score", 0)), float(x.get("recommended_start", x["start"])))
    )

    selected = [
        r for r in records
        if float(r.get("score", 0)) >= MIN_EVENT_SCORE
        and r.get("event_type") != "low_value"
        and str(r.get("visual_evidence", "")).strip()
    ]

    # 如果一条都没选出来，至少导出每个视频的最高分候选，方便人工复查。
    if not selected:
        by_video = {}
        for r in records:
            key = r["source_video"]
            if key not in by_video or float(r.get("score", 0)) > float(by_video[key].get("score", 0)):
                by_video[key] = r
        selected = list(by_video.values())

    selected = sorted(selected, key=lambda x: (x["source_video"], float(x.get("recommended_start", x["start"]))))

    candidate_clips = []
    script = []

    for i, r in enumerate(selected):
        start = float(r.get("recommended_start", r["start"]))
        end = float(r.get("recommended_end", r["end"]))
        timestamp = f"{ts(start)}-{ts(end)}"
        clip_id = f"clip_{i + 1:04d}"
        narration = str(r.get("text", "") or "").strip()
        ost = int(r.get("ost", 1 if not narration else 2))
        if ost in (0, 2) and not narration:
            ost = 1

        candidate_clips.append({
            "clip_id": clip_id,
            "source_event_ids": [f"seg_{int(r['segment_index']):04d}_{r.get('event_type', 'event')}"],
            "timestamp": timestamp,
            "role": r.get("event_type", "unknown"),
            "score": float(r.get("score", 0)),
            "confidence": float(r.get("confidence", 0)),
            "picture": r.get("visual_evidence") or r.get("summary", ""),
            "visual_evidence": r.get("visual_evidence", ""),
            "highlight_reason": r.get("highlight_reason", ""),
            "narration_hint": narration,
            "OST": ost,
            "source_video": r["source_video"],
            "storyboard": r.get("storyboard", ""),
        })

        script.append({
            "_id": clip_id,
            "timestamp": timestamp,
            "picture": r.get("visual_evidence") or r.get("summary", ""),
            "narration": narration,
            "OST": ost,
            "score": float(r.get("score", 0)),
            "event_type": r.get("event_type", "unknown"),
            "visual_evidence": r.get("visual_evidence", ""),
            "source_video": r["source_video"],
        })

    candidate_path = RESULT_DIR / "candidate_clips.json"
    script_path = RESULT_DIR / "narrato_segment_script_converted.json"
    legacy_script_path = RESULT_DIR / "narrato_segment_script.json"

    with open(candidate_path, "w", encoding="utf-8") as f:
        json.dump({"target_duration_seconds": [60, 90], "clips": candidate_clips}, f, ensure_ascii=False, indent=2)

    with open(script_path, "w", encoding="utf-8") as f:
        json.dump(script, f, ensure_ascii=False, indent=2)

    with open(legacy_script_path, "w", encoding="utf-8") as f:
        json.dump(script, f, ensure_ascii=False, indent=2)

    print("原始记录数:", len(records))
    print("候选片段数:", len(candidate_clips))
    print("candidate_clips:", candidate_path)
    print("NarratoAI script:", script_path)
    print(json.dumps(script[:5], indent=2, ensure_ascii=False))
else:
    print("没有新的片段识别结果，跳过 script 导出。")


In [ ]:
%choose narrato_segment_understanding